# 🔍 Joern CPG → Neo4j AuraDB
**Analyze any C++ code with a Code Property Graph, uploaded to Neo4j AuraDB**

---

## 📋 Setup Instructions — Read Before Running Anything

Follow these steps **before** running any cell:

### Step A — Create a Free Neo4j AuraDB Instance

1. Go to **[https://console.neo4j.io](https://console.neo4j.io)** and sign in or create a free account.
2. Click **"New Instance"** → choose the **Free** tier → click **Create**.
3. When the instance is created, a `.txt` credentials file will **automatically download** to your computer. Open it — it contains:

```
NEO4J_USERNAME=neo4j
NEO4J_PASSWORD=<your-password-here>
```

4. To get your **URI**, go back to the AuraDB console, find your instance card, click the **⋮ (three-dot menu)** on the top-right of the card, and choose **"Inspect"**. Copy the **Connection URI** — it looks like:
```
neo4j+s://xxxxxxxx.databases.neo4j.io
```

### Step B — Fill In the Config Cell Below

Paste your credentials into **Cell 1** (the very next cell), then proceed.

### Step C — Paste Your C++ Code

In **Cell 2**, replace the placeholder C++ code with any small C++ program you want to analyze.

### Step D — Run All Cells

Use **Runtime → Run all** (or `Ctrl+F9`). The notebook will install Joern, parse your code, build the CPG, and upload everything to AuraDB automatically.

### Step E — Explore in AuraDB (Last Cell)

After all cells finish, scroll to the **last cell** to find your Cypher queries.
Then:
1. Go to **[https://console.neo4j.io](https://console.neo4j.io)**
2. On your instance card, click the **"Open"** button (or the **"Query"** tab)
3. A Neo4j Browser window opens — paste any Cypher query from the last cell and press **Run ▶**

---

### Pipeline Overview

```
C++ source  (you write it in Cell 2)
    │
    ▼  joern-parse
 .cpg binary
    │
    ▼  joern-export --format neo4jcsv
 CSV + header files
    │
    ▼  direct CSV upload (Python)
    │
    ▼
 Neo4j AuraDB  →  Browser / Bloom
```

In [2]:
!pip install neo4j

In [4]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  CELL 1 — AuraDB Configuration                                  ║
# ║  Fill in all three values from your credentials file & console  ║
# ╚══════════════════════════════════════════════════════════════════╝

# From the .txt file that downloaded when you created the instance:
NEO4J_USER     = 'fce0a5b4'   # e.g. 'neo4j'
NEO4J_PASSWORD = 'hfFZS01Tw4scKEFRT6A1Tt3nEjuS8_xYvpXC5bLr5iE'   # e.g. 'abc123XYZ...'

# From the three-dot menu → Inspect on your instance card:
NEO4J_URI      = 'neo4j+s://fce0a5b4.databases.neo4j.io'   # e.g. 'neo4j+s://xxxxxxxx.databases.neo4j.io'

# ──────────────────────────────────────────────────────────────────
# Quick connectivity check — run this cell to verify credentials
from neo4j import GraphDatabase
_d = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))
_d.verify_connectivity()
_d.close()
print('✅ AuraDB connection OK')

✅ AuraDB connection OK


In [5]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  CELL 2 — Your C++ Code                                         ║
# ║  Replace the code below with any small C++ program to analyze   ║
# ╚══════════════════════════════════════════════════════════════════╝

import pathlib

PYTHON_SOURCE = r"""

import sys
from loguru import logger

_configured = False   # guard against double-initialisation


def setup_logging(
    log_file: str = "student_system.log",
    console_level: str = "DEBUG",
    file_level: str = "DEBUG",
) -> None:

    global _configured
    if _configured:
        return

    # Remove the default Loguru handler
    logger.remove()

    # ── Coloured console output ──────────────────────────────────────────────
    logger.add(
        sys.stderr,
        format=(
            "<green>{time:HH:mm:ss}</green> | "
            "<level>{level: <8}</level> | "
            "<cyan>{name}</cyan>:<cyan>{function}</cyan> - "
            "<level>{message}</level>"
        ),
        level=console_level,
        colorize=True,
    )

    # ── Rotating file log ────────────────────────────────────────────────────
    logger.add(
        log_file,
        format="{time:YYYY-MM-DD HH:mm:ss} | {level: <8} | {name}:{function}:{line} | {message}",
        level=file_level,
        rotation="1 MB",    # rotate when file reaches 1 MB
        retention="7 days", # keep logs for 7 days
        diagnose=False,     # hide variable values in tracebacks (privacy)
        enqueue=True,       # thread-safe writes
    )

    _configured = True
    logger.debug("Loguru logging initialised.")


def get_logger(name: str = "student_system"):

    return logger.bind(name=name)
"""

SRC_DIR = pathlib.Path('/content/src')
SRC_DIR.mkdir(parents=True, exist_ok=True)
(SRC_DIR / 'program.py').write_text(PYTHON_SOURCE)
print('✅ Python source written to /content/src/program.py')
print()
print(PYTHON_SOURCE)

✅ Python source written to /content/src/program.py



import sys
from loguru import logger

_configured = False   # guard against double-initialisation


def setup_logging(
    log_file: str = "student_system.log",
    console_level: str = "DEBUG",
    file_level: str = "DEBUG",
) -> None:

    global _configured
    if _configured:
        return

    # Remove the default Loguru handler
    logger.remove()

    # ── Coloured console output ──────────────────────────────────────────────
    logger.add(
        sys.stderr,
        format=(
            "<green>{time:HH:mm:ss}</green> | "
            "<level>{level: <8}</level> | "
            "<cyan>{name}</cyan>:<cyan>{function}</cyan> - "
            "<level>{message}</level>"
        ),
        level=console_level,
        colorize=True,
    )

    # ── Rotating file log ────────────────────────────────────────────────────
    logger.add(
        log_file,
        format="{time:YYYY-MM-DD HH:mm:ss} | {level: <8} | {name}:{function}:{l

---
## STEP 1 — Install Java 21 (required by Joern)

In [6]:
%%bash
set -e
apt-get update -qq
apt-get install -y -qq wget apt-transport-https gnupg

wget -qO - https://packages.adoptium.net/artifactory/api/gpg/key/public \
  | gpg --dearmor -o /usr/share/keyrings/adoptium.gpg

echo "deb [signed-by=/usr/share/keyrings/adoptium.gpg] \
https://packages.adoptium.net/artifactory/deb \
$(awk -F= '/^VERSION_CODENAME/{print $2}' /etc/os-release) main" \
  > /etc/apt/sources.list.d/adoptium.list

apt-get update -qq
apt-get install -y -qq temurin-21-jdk
java -version
echo '✅ Java 21 installed'

(Reading database ... 122492 files and directories currently installed.)
Preparing to unpack .../wget_1.21.2-2ubuntu1.4_amd64.deb ...
Unpacking wget (1.21.2-2ubuntu1.4) over (1.21.2-2ubuntu1.1) ...
Selecting previously unselected package apt-transport-https.
Preparing to unpack .../apt-transport-https_2.4.14_all.deb ...
Unpacking apt-transport-https (2.4.14) ...
Setting up wget (1.21.2-2ubuntu1.4) ...
Setting up apt-transport-https (2.4.14) ...
Processing triggers for man-db (2.10.2-1) ...
Selecting previously unselected package p11-kit-modules:amd64.
(Reading database ... 122496 files and directories currently installed.)
Preparing to unpack .../0-p11-kit-modules_0.24.0-6build1_amd64.deb ...
Unpacking p11-kit-modules:amd64 (0.24.0-6build1) ...
Selecting previously unselected package p11-kit.
Preparing to unpack .../1-p11-kit_0.24.0-6build1_amd64.deb ...
Unpacking p11-kit (0.24.0-6build1) ...
Selecting previously unselected package adoptium-ca-certificates.
Preparing to unpack .../2-ad

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
openjdk version "21.0.11" 2026-04-21
OpenJDK Runtime Environment (build 21.0.11+10-1-22.04.2-Ubuntu)
OpenJDK 64-Bit Server VM (build 21.0.11+10-1-22.04.2-Ubuntu, mixed mode, sharing)


---
## STEP 2 — Install Joern

In [7]:
%%bash
set -e
cd /opt
wget -q https://github.com/joernio/joern/releases/latest/download/joern-install.sh
chmod +x joern-install.sh
./joern-install.sh --prefix=/opt/joern 2>&1 | tail -10
echo '✅ Joern installed'

  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
100 1736M  100 1736M    0     0  9966k      0  0:02:58  0:02:58 --:--:-- 11.4M
Creating symlinks to Joern tools in /usr/local/bin
If you are not root, please enter your password now:
Installing default plugins
Writing logs to: /tmp/joern-scan-log.txt
[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
Install complete!
✅ Joern installed


In [8]:
import os
os.environ['PATH'] = '/opt/joern:' + os.environ['PATH']
print('✅ PATH updated')

✅ PATH updated


---
## STEP 3 — Install Python dependencies

In [9]:
%%bash
pip install -q neo4j pyyaml
echo '✅ neo4j + pyyaml installed'

✅ neo4j + pyyaml installed


---
## STEP 4 — `joern-parse` → generate CPG binary

In [10]:
!ls -lah /opt/joern/joern-cli

total 216K
drwxr-xr-x  8 root root 4.0K Aug 17 18:14 .
drwxr-xr-x  3 root root 4.0K Aug 17 18:14 ..
-rwxr-xr-x  1 root root  487 Aug 17 08:04 abap2cpg
-rw-r--r--  1 root root  229 Aug 17 08:04 abap2cpg.bat
drwxr-xr-x  2 root root 4.0K Aug 17 18:14 bin
-rw-r--r--  1 root root  223 Aug 17 08:04 c2cpg.bat
-rwxr-xr-x  1 root root  486 Aug 17 08:04 c2cpg.sh
drwxr-xr-x  2 root root 4.0K Aug 17 18:14 conf
-rwxr-xr-x  1 root root  497 Aug 17 08:04 csharpsrc2cpg
-rwxr-xr-x  1 root root  238 Aug 17 08:04 csharpsrc2cpg.bat
drwxr-xr-x 16 root root 4.0K Aug 17 18:14 frontends
-rwxr-xr-x  1 root root  441 Aug 17 08:04 ghidra2cpg
-rw-r--r--  1 root root  233 Aug 17 08:04 ghidra2cpg.bat
-rwxr-xr-x  1 root root  489 Aug 17 08:04 gosrc2cpg
-rw-r--r--  1 root root  231 Aug 17 08:04 gosrc2cpg.bat
-rw-r--r--  1 root root    0 Aug 17 08:04 .installation_root
-rwxr-xr-x  1 root root  492 Aug 17 08:04 javasrc2cpg
-rw-r--r--  1 root root  235 Aug 17 08:04 javasrc2cpg.bat
-rwxr-xr-x  1 root root  490 Aug 17 08:

In [11]:
%%bash
set -e

JOERN_PARSE=$(find /opt/joern -name joern-parse -type f 2>/dev/null | head -1)
JOERN_PARSE=${JOERN_PARSE:-$(command -v joern-parse 2>/dev/null || echo "")}

if [ -z "$JOERN_PARSE" ]; then
  echo "ERROR: joern-parse not found. Did Step 2 complete successfully?"
  exit 1
fi
echo "Using: $JOERN_PARSE"

"$JOERN_PARSE" /content/src \
    --language c \
    --output /content/program.cpg

echo "CPG size: $(du -sh /content/program.cpg)"
echo '✅ Parse complete'

Using: /opt/joern/joern-cli/joern-parse
[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
Parsing code at: /content/src - language: `c`
[+] Running language frontend
Invoking CPG generator in a separate process. Note that the new process will consume additional memory.
If you are importing a large codebase (and/or running into memory issues), please try the following:
1) exit joern
2) invoke the frontend: /opt/joern/joern-cli/c2cpg.sh -J-Xmx3244m /content/src --output /content/program.cpg
3) start joern, import the cpg: `importCpg("path/to/cpg")`

[+] Applying default overlays
Successfully wrote graph to: /content/program.cpg
To load the graph, type `joern /content/program.cpg`
CPG size: 24K	/content/program.cpg

---
## STEP 5 — `joern-export` → neo4jcsv format

This produces **CSV data files** and **header files** that we will upload directly to AuraDB.

In [12]:
%%bash
set -e
rm -rf /content/cpg_csv

JOERN_EXPORT=$(find /opt/joern -name joern-export -type f 2>/dev/null | head -1)
JOERN_EXPORT=${JOERN_EXPORT:-$(command -v joern-export 2>/dev/null || echo "")}

if [ -z "$JOERN_EXPORT" ]; then
  echo "ERROR: joern-export not found."
  exit 1
fi

"$JOERN_EXPORT" /content/program.cpg \
    --out /content/cpg_csv \
    --repr all \
    --format neo4jcsv

echo '✅ Export complete. Files:'
ls -lh /content/cpg_csv/

[0.000s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
exported 11 nodes, 20 edges into /content/cpg_csv
Instructions on how to import the exported files into neo4j:
Prerequisite: ensure you have neo4j community server running (enterprise and desktop may work too)
e.g. download from https://neo4j.com/download-center/#community and start via `bin/neo4j console`

Then, in a new terminal:
```
cd <neo4j_root>

# if you have a fresh instance, you must first change the initial password
bin/cypher-shell -u neo4j -p neo4j
# exit the cypher shell

# copy the data files to the `import` directory, where neo4j will find them
cp /content/cpg_csv/*_data.csv import

find /content/cpg_csv -name 'nodes_*_cypher.csv' -exec bin/cypher-

---
## STEP 6 — Clear AuraDB before import

This wipes the database so you start fresh. Safe to run on every re-run.

In [13]:
from neo4j import GraphDatabase

driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))
with driver.session() as session:
    session.run('MATCH (n) DETACH DELETE n')
driver.close()
print('✅ Database cleared')

✅ Database cleared


---
## STEP 7 — Upload nodes & edges to AuraDB

Reads the exported CSV files directly and pushes all nodes and relationships into AuraDB.

In [14]:
import csv, pathlib, re
from neo4j import GraphDatabase

EXPORT_PATH = pathlib.Path('/content/cpg_csv')
driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))

# ── helpers ────────────────────────────────────────────────────────

def read_raw(path):
    with open(path, newline='', encoding='utf-8') as f:
        return list(csv.reader(f))

def coerce(val):
    if val == '' or val is None:
        return None
    try: return int(val)
    except ValueError: pass
    try: return float(val)
    except ValueError: pass
    return val

def parse_header(header_path):
    rows = read_raw(header_path)
    if not rows:
        return []
    cols = []
    for raw in rows[0]:
        raw = raw.strip()
        if re.match(r'^:?(ID|LABEL|START_ID|END_ID|TYPE)', raw.lstrip(':')):
            cols.append(('__' + raw.lstrip(':').split('(')[0], raw))
        else:
            clean = re.sub(r':.*$', '', raw)
            cols.append((clean, raw))
    return cols

def zip_row(col_defs, row_values):
    return {name: val for (name, _), val in zip(col_defs, row_values)}

# ── node upload ────────────────────────────────────────────────────

def upload_nodes(session, data_file):
    label = re.sub(r'^nodes_|_data$', '', data_file.stem)
    header_file = data_file.parent / data_file.name.replace('_data.csv', '_header.csv')
    col_defs = parse_header(header_file)
    data_rows = read_raw(data_file)
    if not data_rows:
        print(f"  – nodes/{label}: 0 rows, skipped")
        return
    with session.begin_transaction() as tx:
        for raw_row in data_rows:
            row = zip_row(col_defs, raw_row)
            node_id = coerce(row.get('__ID'))
            props = {'_id': node_id}
            for name, val in row.items():
                if name.startswith('__'):
                    continue
                v = coerce(val)
                if v is not None and name:
                    props[name] = v
            tx.run(f"CREATE (n:`{label}`) SET n = $props", props=props)
        tx.commit()
    print(f"  ✓ nodes/{label}: {len(data_rows)} rows")

# ── edge upload ────────────────────────────────────────────────────

def upload_edges(session, data_file):
    rel_type = re.sub(r'^edges_|_data$', '', data_file.stem)
    header_file = data_file.parent / data_file.name.replace('_data.csv', '_header.csv')
    col_defs = parse_header(header_file)
    data_rows = read_raw(data_file)
    if not data_rows:
        return
    with session.begin_transaction() as tx:
        for raw_row in data_rows:
            row = zip_row(col_defs, raw_row)
            src = coerce(row.get('__START_ID'))
            dst = coerce(row.get('__END_ID'))
            if src is None or dst is None:
                continue
            props = {}
            for name, val in row.items():
                if name.startswith('__') or not name:
                    continue
                v = coerce(val)
                if v is not None:
                    props[name] = v
            tx.run(
                f"MATCH (a {{_id:$s}}), (b {{_id:$d}}) "
                f"MERGE (a)-[r:`{rel_type}`]->(b) SET r += $props",
                s=src, d=dst, props=props)
        tx.commit()
    print(f"  ✓ edges/{rel_type}: {len(data_rows)} rows")

# ── run ────────────────────────────────────────────────────────────

print("=== Uploading nodes ===")
with driver.session() as session:
    for data_file in sorted(EXPORT_PATH.glob('nodes_*_data.csv')):
        try:
            upload_nodes(session, data_file)
        except Exception as e:
            print(f"  ✗ {data_file.name}: {e}")

print("\n=== Uploading edges ===")
with driver.session() as session:
    for data_file in sorted(EXPORT_PATH.glob('edges_*_data.csv')):
        try:
            upload_edges(session, data_file)
        except Exception as e:
            print(f"  ✗ {data_file.name}: {e}")

driver.close()
print("\n✅ Upload complete")

=== Uploading nodes ===
  ✓ nodes/BLOCK: 1 rows
  ✓ nodes/FILE: 2 rows
  ✓ nodes/META_DATA: 1 rows
  ✓ nodes/METHOD_RETURN: 1 rows
  ✓ nodes/METHOD: 1 rows
  ✓ nodes/NAMESPACE_BLOCK: 2 rows
  ✓ nodes/NAMESPACE: 1 rows
  ✓ nodes/TYPE_DECL: 1 rows
  ✓ nodes/TYPE: 1 rows

=== Uploading edges ===
  ✓ edges/AST: 5 rows
  ✓ edges/CFG: 1 rows
  ✓ edges/CONTAINS: 3 rows
  ✓ edges/DOMINATE: 1 rows
  ✓ edges/EVAL_TYPE: 2 rows
  ✓ edges/POST_DOMINATE: 1 rows
  ✓ edges/REF: 3 rows
  ✓ edges/SOURCE_FILE: 4 rows

✅ Upload complete


---
## STEP 8 — Verify: count nodes & edges in AuraDB

In [15]:
from neo4j import GraphDatabase

driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))
with driver.session() as s:
    nodes = s.run('MATCH (n) RETURN count(n) AS c').single()['c']
    edges = s.run('MATCH ()-[r]->() RETURN count(r) AS c').single()['c']
    print(f'Nodes : {nodes}')
    print(f'Edges : {edges}')

    print('\nNode labels:')
    for r in s.run('MATCH (n) RETURN DISTINCT labels(n) AS l, count(n) AS c ORDER BY c DESC'):
        print(f'  {r["l"]} → {r["c"]}')

    print('\nEdge types:')
    for r in s.run('MATCH ()-[r]->() RETURN DISTINCT type(r) AS t, count(r) AS c ORDER BY c DESC'):
        print(f'  {r["t"]} → {r["c"]}')
driver.close()

Nodes : 11
Edges : 20

Node labels:
  ['FILE'] → 2
  ['NAMESPACE_BLOCK'] → 2
  ['BLOCK'] → 1
  ['META_DATA'] → 1
  ['METHOD_RETURN'] → 1
  ['METHOD'] → 1
  ['NAMESPACE'] → 1
  ['TYPE_DECL'] → 1
  ['TYPE'] → 1

Edge types:
  AST → 5
  SOURCE_FILE → 4
  CONTAINS → 3
  REF → 3
  EVAL_TYPE → 2
  POST_DOMINATE → 1
  CFG → 1
  DOMINATE → 1


# STEP 9 — Run Cypher Queries in Neo4j Browser

## How to use these queries

1. Go to :contentReference[oaicite:0]{index=0}
2. Open your AuraDB instance
3. Click the **Query** tab to open **Neo4j Browser**
4. Copy any Cypher query from the next cell
5. Paste it into the query editor
6. Press **Run ▶**

---

## Important

After running the queries, Neo4j Browser will generate **graph visualizations/images** showing:

- AST (Abstract Syntax Tree)
- CFG (Control Flow Graph)
- PDG (Program Dependence Graph)
- Method relationships
- Call graphs
- Taint/data flow paths
- Other structural program analysis outputs

These generated graph visualizations are part of the **lab results** and should be:

- captured as screenshots/images
- included in your final lab report
- explained briefly in the report

Different queries generate different graph structures, so include multiple meaningful outputs in the report.

# Cypher Queries — Joern CPG in Neo4j AuraDB

Copy any query below directly into Neo4j Browser.

---

## Query 1 — All nodes & edges (overview)

Visualize the entire graph.

```cypher
MATCH (n)-[r]->(m)
RETURN n, r, m
LIMIT 300
```

---

## Query 2 — AST (Abstract Syntax Tree)

Shows the hierarchical structure of the source code.

```cypher
MATCH p=(root)-[:AST*]->(leaf)
WHERE NOT ()-[:AST]->(root)
RETURN p
LIMIT 300
```

---

## Query 3 — CFG (Control Flow Graph)

Shows execution order between statements.

```cypher
MATCH p=()-[:CFG*1..10]->()
RETURN p
LIMIT 200
```

---

## Query 4 — PDG (Program Dependence Graph)

Shows data and control dependencies.

```cypher
MATCH p=()-[:REACHING_DEF|CDG*1..5]->()
RETURN p
LIMIT 200
```

